In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# ── reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
# 1. LOAD & SORT
# ─────────────────────────────────────────────────────────────────────────────
df = pd.read_csv("final_features_GOOGL_2022_2025.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

TARGET_COL = "target_next_return"

id_cols     = ["ticker", "date"]
target_cols = ["target_next_return", "target_next_price", "target_5d_return"]

# BUG FIX: simple_return is perfectly collinear with return_lag_1.shift(-1).
# It is NOT a leak (today's return is a valid known feature), but it is
# redundant and causes multicollinearity. Drop it.
feature_cols = [c for c in df.columns if c not in id_cols + target_cols + ["simple_return"]]

print(f"Shape      : {df.shape}")
print(f"Target     : {TARGET_COL}")
print(f"# Features : {len(feature_cols)}")
print(f"Features   : {feature_cols}")
print(f"Tickers    : {df['ticker'].nunique()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device     : {device}")

test_preds = []

# ─────────────────────────────────────────────────────────────────────────────
# 2. METRICS
# ─────────────────────────────────────────────────────────────────────────────
def directional_accuracy(y_true, y_pred):
    return (np.sign(y_true) == np.sign(y_pred)).mean()

def sharpe_ratio_from_predictions(y_true, y_pred, annualization=252):
    positions        = np.sign(y_pred)
    strategy_returns = positions * y_true
    std              = np.std(strategy_returns)
    return np.nan if std == 0 else (np.mean(strategy_returns) / std) * np.sqrt(annualization)

def regression_metrics(y_true, y_pred, include_sharpe=True):
    mse = mean_squared_error(y_true, y_pred)
    metrics = {
        "MSE"                 : mse,
        "RMSE"                : np.sqrt(mse),
        "MAE"                 : mean_absolute_error(y_true, y_pred),
        "R2"                  : r2_score(y_true, y_pred),
        "Directional_Accuracy": directional_accuracy(y_true, y_pred),
    }
    if include_sharpe:
        metrics["Sharpe"] = sharpe_ratio_from_predictions(
            np.array(y_true), np.array(y_pred)
        )
    return metrics

# ─────────────────────────────────────────────────────────────────────────────
# 3. PER-TICKER NORMALIZATION + SEQUENCE BUILDING
# ─────────────────────────────────────────────────────────────────────────────
SEQ_LEN = 20   # look-back window (trading days)

def build_sequences_for_group(X_vals, y_vals, dates, seq_len):
    """Slide a window over one ticker's data → (n_samples, seq_len, n_features)."""
    Xs, ys, ds = [], [], []
    for i in range(seq_len, len(X_vals)):
        Xs.append(X_vals[i - seq_len : i])
        ys.append(y_vals[i])
        ds.append(dates[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32), np.array(ds)


def build_scaled_sequences(df, feature_cols, target_col, seq_len,
                            train_months, val_months, test_months):
    X_train_all, y_train_all = [], []
    X_val_all,   y_val_all   = [], []
    X_test_all,  y_test_all  = [], []
    meta_train, meta_val, meta_test = [], [], []

    train_ym = {(p.year, p.month) for p in train_months}
    val_ym   = {(p.year, p.month) for p in val_months}
    test_ym  = {(p.year, p.month) for p in test_months}

    skipped = []
    for ticker, group in df.groupby("ticker"):
        group = group.sort_values("date").reset_index(drop=True)

        ym_series = list(zip(group["date"].dt.year, group["date"].dt.month))
        tr_mask   = np.array([t in train_ym for t in ym_series])
        val_mask  = np.array([t in val_ym   for t in ym_series])
        test_mask = np.array([t in test_ym  for t in ym_series])

        n_train = tr_mask.sum()
        n_val   = val_mask.sum()
        n_test  = test_mask.sum()

        reason = None
        if n_train < seq_len + 1:
            reason = f"train only {n_train} rows (need {seq_len+1})"
        elif n_val == 0:
            reason = "no val rows"
        elif n_test == 0:
            reason = "no test rows"

        if reason:
            skipped.append((ticker, reason))
            continue

        scaler = StandardScaler()
        X_full = group[feature_cols].values.astype(np.float32)
        y_full = group[target_col].values.astype(np.float32)
        dates  = group["date"].values

        scaler.fit(X_full[tr_mask])
        X_scaled = scaler.transform(X_full)

        Xs_full, ys_full, ds_full = build_sequences_for_group(
            X_scaled, y_full, dates, seq_len
        )

        target_ym = [(pd.Timestamp(d).year, pd.Timestamp(d).month) for d in ds_full]

        def _collect(ym_set):
            mask = np.array([t in ym_set for t in target_ym])
            return Xs_full[mask], ys_full[mask], ds_full[mask]

        Xtr, ytr, dtr = _collect(train_ym)
        Xva, yva, dva = _collect(val_ym)
        Xte, yte, dte = _collect(test_ym)

        X_train_all.append(Xtr);  y_train_all.append(ytr)
        X_val_all.append(Xva);    y_val_all.append(yva)
        X_test_all.append(Xte);   y_test_all.append(yte)

        for d in dtr: meta_train.append((ticker, d))
        for d in dva: meta_val.append((ticker, d))
        for d in dte: meta_test.append((ticker, d))

    if skipped:
        print(f"  [skip] {len(skipped)} ticker(s) dropped this fold:")
        for tkr, rsn in skipped:
            print(f"         {tkr}: {rsn}")

    def _stack(lst):
        valid = [a for a in lst if len(a) > 0]
        return (np.concatenate(valid, axis=0) if valid
                else np.empty((0, seq_len, len(feature_cols)), dtype=np.float32))

    def _cat(lst):
        valid = [a for a in lst if len(a) > 0]
        return (np.concatenate(valid, axis=0) if valid
                else np.empty((0,), dtype=np.float32))

    X_train = _stack(X_train_all); y_train = _cat(y_train_all)
    X_val   = _stack(X_val_all);   y_val   = _cat(y_val_all)
    X_test  = _stack(X_test_all);  y_test  = _cat(y_test_all)

    y_mean = y_train.mean()
    y_std  = y_train.std() + 1e-8

    y_train_n = (y_train - y_mean) / y_std
    y_val_n   = (y_val   - y_mean) / y_std
    # y_test is intentionally NOT normalized — we denormalize predictions instead

    meta_train_df = pd.DataFrame(meta_train, columns=["ticker", "date"])
    meta_val_df   = pd.DataFrame(meta_val,   columns=["ticker", "date"])
    meta_test_df  = pd.DataFrame(meta_test,  columns=["ticker", "date"])

    return (X_train, y_train_n, meta_train_df,
            X_val,   y_val_n,   meta_val_df,
            X_test,  y_test,    meta_test_df,
            y_mean,  y_std)

# ─────────────────────────────────────────────────────────────────────────────
# 4. ROLLING-WINDOW FOLDS
# ─────────────────────────────────────────────────────────────────────────────
all_months = sorted(df["date"].dt.to_period("M").unique())
print(f"Months in dataset: {all_months[0]} → {all_months[-1]}  ({len(all_months)} total)")

TRAIN_WINDOW = 12  # months

def make_rolling_folds(all_months, train_window=12):
    """
    Returns list of (train_months, val_month, test_month) tuples.

    Layout per fold (strictly forward-looking, no leakage):
      train : [i  …  i+train_window-1]   (12 months)
      val   : [i+train_window]            (1 month)
      test  : [i+train_window+1]          (1 month)
    """
    folds = []
    for i in range(len(all_months) - train_window - 1):
        train_months = list(all_months[i : i + train_window])
        val_month    = [all_months[i + train_window]]
        test_month   = [all_months[i + train_window + 1]]
        folds.append((train_months, val_month, test_month))
    return folds

rolling_folds = make_rolling_folds(all_months, TRAIN_WINDOW)

print(f"\nRolling folds ({len(rolling_folds)} total):")
for k, (tr, va, te) in enumerate(rolling_folds):
    print(f"  Fold {k+1:>3}: train={str(tr[0])}→{str(tr[-1])}  val={str(va[0])}  test={str(te[0])}")

# ── Final production prediction window ───────────────────────────────────────
PREDICT_MONTH    = pd.Period("2025-12", freq="M")
pred_train_end   = PREDICT_MONTH - 1
pred_train_start = pred_train_end - (TRAIN_WINDOW - 1)

pred_train_months = [m for m in all_months
                     if pred_train_start <= m <= pred_train_end]
pred_target_month = [PREDICT_MONTH]

print("\n── Final prediction window ──────────────────────────────────────────────")
print(f"  Train : {str(pred_train_months[0])} → {str(pred_train_months[-1])}  ({len(pred_train_months)} months)")
print(f"  Predict on : {str(pred_target_month[0])}")

# ─────────────────────────────────────────────────────────────────────────────
# 5. TORCH HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def make_loader(X, y, batch_size=64, shuffle=False):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32).view(-1, 1),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      pin_memory=(device.type == "cuda"))


def train_model(model, train_loader, val_loader,
                epochs=30, lr=1e-3, patience=5):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=3, factor=0.5
    )

    best_val_loss = np.inf
    best_state    = None
    no_improve    = 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_losses.append(criterion(model(xb), yb).item())

        mean_val = np.mean(val_losses)
        scheduler.step(mean_val)

        if mean_val < best_val_loss:
            best_val_loss = mean_val
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1

        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1:>3}/{epochs} | val_loss={mean_val:.6f}"
                  + (" ✓" if no_improve == 0 else ""))

        if no_improve >= patience:
            print(f"  Early stop at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)
    return model


def predict_model(model, X, batch_size=256):
    model.eval()
    preds = []
    loader = DataLoader(torch.tensor(X, dtype=torch.float32),
                        batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for xb in loader:
            preds.extend(model(xb.to(device)).cpu().numpy().ravel())
    return np.array(preds)


# ─────────────────────────────────────────────────────────────────────────────
# 6. MODEL DEFINITIONS
# ─────────────────────────────────────────────────────────────────────────────
n_features = len(feature_cols)

# ── 6a. TCN ──────────────────────────────────────────────────────────────────
class CausalConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size,
                              dilation=dilation, padding=self.padding)

    def forward(self, x):
        return self.conv(x)[:, :, : x.size(2)]


class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(in_ch, out_ch, kernel_size, dilation),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.Dropout(dropout),
            CausalConv1d(out_ch, out_ch, kernel_size, dilation),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.residual = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        return nn.functional.gelu(self.net(x) + self.residual(x))


class TCN(nn.Module):
    def __init__(self, n_features, num_channels=(64, 128, 64),
                 kernel_size=3, dropout=0.2):
        super().__init__()
        layers = []
        in_ch = n_features
        for i, out_ch in enumerate(num_channels):
            dilation = 2 ** i
            layers.append(TCNBlock(in_ch, out_ch, kernel_size, dilation, dropout))
            in_ch = out_ch
        self.tcn  = nn.Sequential(*layers)
        self.head = nn.Linear(in_ch, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.tcn(x)
        x = x[:, :, -1]
        return self.head(x)


# ── 6b. LSTM ─────────────────────────────────────────────────────────────────
class LSTMModel(nn.Module):
    def __init__(self, n_features, hidden=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers=num_layers,
                            batch_first=True, dropout=dropout)
        self.norm = nn.LayerNorm(hidden)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(32, 1))

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(self.norm(out[:, -1, :]))


# ── 6c. Transformer ──────────────────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, : x.size(1)])


class TransformerModel(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4,
                 num_layers=2, dropout=0.1):
        super().__init__()
        assert d_model % nhead == 0, "d_model must be divisible by nhead"
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_enc    = PositionalEncoding(d_model, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.Linear(d_model, 32), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(32, 1))

    def forward(self, x):
        x = self.pos_enc(self.input_proj(x))
        x = self.encoder(x)
        return self.head(self.norm(x[:, -1, :]))


# ── 6d. CNN + LSTM ───────────────────────────────────────────────────────────
class CNNLSTMModel(nn.Module):
    def __init__(self, n_features, cnn_channels=64, lstm_hidden=128,
                 kernel_size=3, dropout=0.2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(n_features, cnn_channels, kernel_size, padding=kernel_size//2),
            nn.BatchNorm1d(cnn_channels),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(cnn_channels, cnn_channels, kernel_size, padding=kernel_size//2),
            nn.BatchNorm1d(cnn_channels),
            nn.GELU(),
        )
        self.lstm = nn.LSTM(cnn_channels, lstm_hidden, num_layers=2,
                            batch_first=True, dropout=dropout)
        self.norm = nn.LayerNorm(lstm_hidden)
        self.head = nn.Sequential(nn.Linear(lstm_hidden, 32), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(32, 1))

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        return self.head(self.norm(out[:, -1, :]))

# ── 6e. TFT ────────────────────────────────────────────────────────────────
class GatedResidualNetwork(nn.Module):
    def __init__(self, dim, hidden_dim=None, dropout=0.1):
        super().__init__()
        hidden_dim = hidden_dim or dim
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(dim)
        self.gate = nn.Linear(dim, dim)

    def forward(self, x):
        residual = x
        x = torch.relu(self.fc1(x))
        x = self.dropout(self.fc2(x))
        gate = torch.sigmoid(self.gate(residual))
        return self.norm(gate * x + (1 - gate) * residual)


class VariableSelectionNetwork(nn.Module):
    def __init__(self, n_features, d_model):
        super().__init__()
        self.weights = nn.Linear(n_features, n_features)
        self.proj = nn.Linear(n_features, d_model)

    def forward(self, x):
        # x: (B, T, F)
        w = torch.softmax(self.weights(x.mean(dim=1)), dim=-1)  # (B, F)
        x = x * w.unsqueeze(1)
        return self.proj(x)


class TFTModel(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4,
                 num_layers=2, dropout=0.1):
        super().__init__()

        self.vsn = VariableSelectionNetwork(n_features, d_model)

        self.lstm = nn.LSTM(
            d_model, d_model,
            num_layers=1,
            batch_first=True
        )

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers)

        self.grn = GatedResidualNetwork(d_model, dropout=dropout)
        self.norm = nn.LayerNorm(d_model)

        self.head = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        # x: (B, T, F)
        x = self.vsn(x)
        x, _ = self.lstm(x)
        x = self.transformer(x)
        x = self.grn(x)
        x = self.norm(x[:, -1, :])
        return self.head(x)


# ─────────────────────────────────────────────────────────────────────────────
# 7. MODEL FACTORY
# ─────────────────────────────────────────────────────────────────────────────
def get_model(name, n_features):
    if name == "TCN":
        return TCN(n_features, num_channels=(64, 128, 64), kernel_size=3, dropout=0.2)
    if name == "LSTM":
        return LSTMModel(n_features, hidden=128, num_layers=2, dropout=0.2)
    if name == "Transformer":
        return TransformerModel(n_features, d_model=64, nhead=4,
                                num_layers=2, dropout=0.1)
    if name == "CNN_LSTM":
        return CNNLSTMModel(n_features, cnn_channels=64, lstm_hidden=128,
                            kernel_size=3, dropout=0.2)
    if name == "TFT":
        return TFTModel(n_features, d_model=64, nhead=4,
                        num_layers=2, dropout=0.1)
    raise ValueError(f"Unknown model: {name}")


MODEL_NAMES = ["TCN", "LSTM", "Transformer", "CNN_LSTM", "TFT"]

# ─────────────────────────────────────────────────────────────────────────────
# 8. ROLLING-WINDOW TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
EPOCHS     = 50
BATCH_SIZE = 128
LR         = 1e-3
PATIENCE   = 7

all_results = []

for fold_idx, (train_months, val_months, test_months) in enumerate(rolling_folds):
    print(f"\n{'='*65}")
    print(f"FOLD {fold_idx+1}/{len(rolling_folds)} | "
          f"train={train_months[0]}→{train_months[-1]} | "
          f"val={val_months[0]} | test={test_months[0]}")
    print("="*65)

    (X_train, y_train, meta_train,
     X_val,   y_val,   meta_val,
     X_test,  y_test,  meta_test,
     y_mean,  y_std) = build_scaled_sequences(
        df, feature_cols, TARGET_COL, SEQ_LEN,
        train_months, val_months, test_months
    )

    if len(X_train) == 0 or len(X_val) == 0 or len(X_test) == 0:
        print("  Skipping fold — insufficient data.")
        continue

    print(f"  Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

    train_loader = make_loader(X_train, y_train, BATCH_SIZE, shuffle=True)
    val_loader   = make_loader(X_val,   y_val,   BATCH_SIZE, shuffle=False)

    for model_name in MODEL_NAMES:
        print(f"\n  ── {model_name} ──")
        model = get_model(model_name, n_features)
        model = train_model(model, train_loader, val_loader,
                            epochs=EPOCHS, lr=LR, patience=PATIENCE)

        y_pred_norm = predict_model(model, X_test)
        y_pred      = y_pred_norm * y_std + y_mean  # denormalize

        for i in range(len(y_test)):
            test_preds.append({
                "date":   meta_test.iloc[i]["date"],
                "ticker": meta_test.iloc[i]["ticker"],
                "fold":   fold_idx + 1,
                "model":  model_name,
                "y_true": float(y_test[i]),
                "y_pred": float(y_pred[i]),
            })

        m = regression_metrics(y_test, y_pred)

        # BUG FIX: was referencing undefined train_years/val_years/test_years
        row = {
            "fold"        : fold_idx + 1,
            "train_period": f"{train_months[0]}→{train_months[-1]}",
            "val_month"   : str(val_months[0]),
            "test_month"  : str(test_months[0]),
            "model"       : model_name,
            **m,
        }
        all_results.append(row)

        print(f"    RMSE={m['RMSE']:.5f}  MAE={m['MAE']:.5f}  "
              f"R2={m['R2']:.4f}  DA={m['Directional_Accuracy']:.3f}  "
              f"Sharpe={m['Sharpe']:.3f}")

# ─────────────────────────────────────────────────────────────────────────────
# 9. RESULTS SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
results_df = pd.DataFrame(all_results)
results_df.to_csv("rolling_window_results.csv", index=False)

metric_cols = ["RMSE", "MAE", "R2", "Directional_Accuracy", "Sharpe"]
summary = (results_df
           .groupby("model")[metric_cols]
           .agg(["mean", "std"])
           .round(5))

print("\n" + "="*65)
print("SUMMARY — averaged across all folds")
print("="*65)
print(summary.to_string())

print("\n── Rankings (best model per metric) ──")
mean_summary = results_df.groupby("model")[metric_cols].mean()
for m in metric_cols:
    if m in ("R2", "Directional_Accuracy", "Sharpe"):
        best = mean_summary[m].idxmax()
    else:
        best = mean_summary[m].idxmin()
    print(f"  {m:<25}: {best}  ({mean_summary.loc[best, m]:.5f})")

pred_df = pd.DataFrame(test_preds)
pred_df = pred_df.sort_values(["date", "ticker", "model"]).reset_index(drop=True)

pred_df.to_csv("dl_test_predictions.csv", index=False)
print("Saved DL test predictions → dl_test_predictions.csv")

Shape      : (998, 19)
Target     : target_next_return
# Features : 13
Features   : ['return_lag_1', 'return_lag_2', 'return_lag_5', 'return_lag_10', 'volume_lag_1', 'volume_lag_5', 'roc_5', 'roc_10', 'roc_20', 'ewm_vol_10', 'relative_return_1', 'relative_strength_5', 'relative_strength_20']
Tickers    : 1
Device     : cpu
Months in dataset: 2022-01 → 2025-12  (48 total)

Rolling folds (35 total):
  Fold   1: train=2022-01→2022-12  val=2023-01  test=2023-02
  Fold   2: train=2022-02→2023-01  val=2023-02  test=2023-03
  Fold   3: train=2022-03→2023-02  val=2023-03  test=2023-04
  Fold   4: train=2022-04→2023-03  val=2023-04  test=2023-05
  Fold   5: train=2022-05→2023-04  val=2023-05  test=2023-06
  Fold   6: train=2022-06→2023-05  val=2023-06  test=2023-07
  Fold   7: train=2022-07→2023-06  val=2023-07  test=2023-08
  Fold   8: train=2022-08→2023-07  val=2023-08  test=2023-09
  Fold   9: train=2022-09→2023-08  val=2023-09  test=2023-10
  Fold  10: train=2022-10→2023-09  val=2023-10  te

/var/folders/_h/b34996m51sb595dtyp0k7l300000gn/T/ipykernel_48935/136971620.py:403: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


  Epoch   5/50 | val_loss=0.985064
  Epoch  10/50 | val_loss=1.022028
  Early stop at epoch 10
    RMSE=0.03106  MAE=0.02157  R2=-0.0150  DA=0.632  Sharpe=1.660

  ── CNN_LSTM ──
  Epoch   5/50 | val_loss=0.901145
  Early stop at epoch 8
    RMSE=0.03139  MAE=0.02168  R2=-0.0367  DA=0.526  Sharpe=0.177

  ── TFT ──


/var/folders/_h/b34996m51sb595dtyp0k7l300000gn/T/ipykernel_48935/136971620.py:494: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, num_layers)


  Epoch   5/50 | val_loss=0.765619
  Early stop at epoch 8
    RMSE=0.03157  MAE=0.02218  R2=-0.0483  DA=0.421  Sharpe=-2.611

FOLD 2/35 | train=2022-02→2023-01 | val=2023-02 | test=2023-03
  Train: (251, 20, 13)  Val: (19, 20, 13)  Test: (23, 20, 13)

  ── TCN ──
  Epoch   5/50 | val_loss=1.646100 ✓
  Epoch  10/50 | val_loss=1.707068
  Early stop at epoch 12
    RMSE=0.01979  MAE=0.01642  R2=-0.0719  DA=0.609  Sharpe=4.712

  ── LSTM ──
  Epoch   5/50 | val_loss=1.584490 ✓
  Epoch  10/50 | val_loss=1.620884
  Epoch  15/50 | val_loss=1.648334
  Early stop at epoch 15
    RMSE=0.02000  MAE=0.01631  R2=-0.0944  DA=0.652  Sharpe=5.234

  ── Transformer ──


...

  Epoch   5/50 | val_loss=0.777837
  Early stop at epoch 9
    RMSE=0.02467  MAE=0.02046  R2=-0.0056  DA=0.526  Sharpe=3.743

FOLD 35/35 | train=2024-11→2025-10 | val=2025-11 | test=2025-12
  Train: (250, 20, 13)  Val: (19, 20, 13)  Test: (17, 20, 13)

  ── TCN ──
  Epoch   5/50 | val_loss=1.483677 ✓
  Epoch  10/50 | val_loss=1.451810 ✓
  Epoch  15/50 | val_loss=1.385183 ✓
  Epoch  20/50 | val_loss=1.324664 ✓
  Epoch  25/50 | val_loss=1.436947
  Early stop at epoch 28
    RMSE=0.01692  MAE=0.01427  R2=-0.3039  DA=0.294  Sharpe=-9.919

  ── LSTM ──
  Epoch   5/50 | val_loss=1.551218
  Epoch  10/50 | val_loss=1.593638
  Early stop at epoch 10
    RMSE=0.01575  MAE=0.01247  R2=-0.1298  DA=0.529  Sharpe=-0.001

  ── Transformer ──


/var/folders/_h/b34996m51sb595dtyp0k7l300000gn/T/ipykernel_48935/136971620.py:403: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


  Epoch   5/50 | val_loss=1.467642
  Epoch  10/50 | val_loss=1.430128
  Epoch  15/50 | val_loss=1.443099
  Early stop at epoch 16
    RMSE=0.01588  MAE=0.01256  R2=-0.1480  DA=0.529  Sharpe=-0.001

  ── CNN_LSTM ──
  Epoch   5/50 | val_loss=1.514666
  Early stop at epoch 8
    RMSE=0.01557  MAE=0.01229  R2=-0.1047  DA=0.529  Sharpe=-0.001

  ── TFT ──


/var/folders/_h/b34996m51sb595dtyp0k7l300000gn/T/ipykernel_48935/136971620.py:494: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, num_layers)


  Epoch   5/50 | val_loss=1.463514
  Early stop at epoch 9
    RMSE=0.01524  MAE=0.01225  R2=-0.0582  DA=0.529  Sharpe=-0.001

SUMMARY — averaged across all folds
                RMSE               MAE                R2          Directional_Accuracy            Sharpe         
                mean      std     mean      std     mean      std                 mean      std     mean      std
model                                                                                                            
CNN_LSTM     0.01867  0.00532  0.01400  0.00371 -0.08242  0.14500              0.53315  0.10226  1.14914  3.56559
LSTM         0.01857  0.00543  0.01396  0.00375 -0.06222  0.08699              0.54738  0.09337  1.70186  3.11425
TCN          0.01849  0.00530  0.01391  0.00355 -0.05729  0.09789              0.54449  0.11883  1.27497  4.21068
TFT          0.01843  0.00530  0.01383  0.00363 -0.04917  0.06839              0.52058  0.10521  0.42783  3.65806
Transformer  0.01874  0.00535  0.01403 